<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Las Limitaciones de los Modelos Recurrentes</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Las arquitecturas recurrentes fueron muy útiles, pero también arrastran límites importantes cuando las secuencias crecen o las dependencias se vuelven largas. Este notebook hace explícitas esas restricciones.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Entender las limitaciones de una familia de modelos explica mejor por qué surgieron mecanismos de atención y transformers.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A reconocer los principales problemas prácticos de los modelos recurrentes al trabajar con texto.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Conocer dónde falla una arquitectura es tan importante como saber dónde funciona bien.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Las RNN pueden sufrir dificultades para retener dependencias largas y para paralelizar el procesamiento de secuencias.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Secuencia larga para observar gradientes
"""Este bloque define una secuencia extensa para ilustrar cómo los modelos recurrentes presentan dificultades al manejar dependencias largas.
Este bloque importa "torch" para trabajar con tensores en PyTorch.

Aplicamos la función "torch.randint()" a la variable "secuencia_larga" para generar un tensor de números enteros aleatorios que simulan 
los índices de palabras en una frase extensa.
El parámetro "low=" indica el valor mínimo incluido que puede tomar cada entero generado.
El parámetro "high=" indica el límite superior excluido de los enteros generados.
El parámetro "size=" define la forma del tensor resultante, en este caso un tensor de 1 fila y 200 columnas.

La variable "secuencia_larga" contiene un tensor de índices que simulan una frase muy extensa.
Esta estructura permite examinar un gradiente asociado a una secuencia prolongada."""

import torch

secuencia_larga = torch.randint(low=1, high=50, size=(1, 200))

secuencia_larga # Muestra la secuencia larga generada para ver su contenido.


In [ ]:
# Bloque 2 — Capa de embeddings
"""Este bloque define una capa de embeddings para transformar índices en vectores densos.
Este bloque importa "torch.nn" para definir modelos de redes neuronales recurrentes.
La variable "embedding_dim" se establece en 32, lo que significa que cada índice se representará en un vector denso de 32 dimensiones.

Aplicamos la clase "nn.Embedding()" a la variable "capa_embeddings" para crear la capa de embeddings que transformará índices en vectores densos.
El parámetro "num_embeddings=" indica el número de índices distintos que se pueden representar; en este caso, 50.
El parámetro "embedding_dim=" indica la dimensionalidad de los vectores densos; en este caso, el valor de "embedding_dim", es decir, 32.
La variable "capa_embeddings" almacena una instancia de la clase que se utilizará para convertir índices de palabras en vectores densos.

Asignamos la llamada a la instancia de la clase a la variable "embeddings" mediante la expresión "capa_embeddings(secuencia_larga)".
El argumento "secuencia_larga" representa la secuencia de índices de palabras que se desea convertir en vectores densos.
La variable "embeddings" almacena los vectores densos correspondientes a la secuencia de índices de palabras, inicializados sin entrenamiento."""

import torch.nn as nn

embedding_dim = 32

capa_embeddings = nn.Embedding(num_embeddings=50, embedding_dim=embedding_dim)

embeddings = capa_embeddings(secuencia_larga)

embeddings # Muestra los vectores densos correspondientes a la secuencia de índices.


In [ ]:
# Bloque 3 — Definición de una RNN simple
"""Este bloque define una RNN simple que procesará la secuencia larga.
La variable "hidden_size" se establece en 64, lo que significa que el estado oculto de la RNN tendrá 64 dimensiones.

Aplicamos la clase "nn.RNN()" a la variable "rnn" para definir una red neuronal recurrente simple.
El parámetro "input_size=" indica la dimensionalidad de los embeddings de entrada; en este caso, el valor de "embedding_dim", es decir, 32 dimensiones.
El parámetro "hidden_size=" define la dimensionalidad del estado oculto; en este caso, el valor de "hidden_size", es decir, 64 dimensiones.
El parámetro "batch_first=" indica si la primera dimensión de la entrada y la salida corresponde al tamaño del batch; en este caso, se establece como "True".
La variable "rnn" almacena una instancia del modelo recurrente que procesará la secuencia de entrada."""

hidden_size = 64

rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_size, batch_first=True)

rnn # Muestra las características de la arquitectura de la RNN definida.


In [ ]:
# Bloque 4 — Forward pass para observar la acumulación secuencial
"""Este bloque procesa la secuencia mediante la RNN.

Aplicamos la función "torch.zeros()" a la variable "estado_inicial" para inicializar el estado oculto con ceros en todas sus dimensiones.
El primer argumento "1" indica el número de capas y direcciones de la RNN en este caso.
El segundo argumento "1" indica el tamaño del lote.
El argumento "hidden_size" indica la dimensionalidad del estado oculto, en este caso 64 dimensiones.
La variable "estado_inicial" almacena el estado oculto inicial de la RNN, que en este caso es un tensor de ceros en todas sus dimensiones.

Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase a las variables "salida_rnn" y "estado_final" mediante la expresión "rnn(embeddings, estado_inicial)".
El argumento "embeddings" es el conjunto de tensores densos de entrada que queremos procesar mediante la RNN.
El argumento "estado_inicial" es el estado oculto de partida definido previamente para la RNN.

La variable "salida_rnn" contiene las salidas generadas en cada paso en forma de tensor.
La variable "estado_final" contiene el estado oculto final tras procesar toda la secuencia.
La dependencia entre pasos limita su paralelización temporal, aunque otras operaciones pueden paralelizarse."""

estado_inicial = torch.zeros(1, 1, hidden_size)

salida_rnn, estado_final = rnn(embeddings, estado_inicial)

salida_rnn, estado_final  # Muestra los resultados de la RNN: las salidas de cada paso y el estado final.


In [ ]:
# Bloque 5 — Cálculo del gradiente inicial
"""Este bloque calcula el gradiente del estado final respecto a los embeddings iniciales.
Aplicamos el método ".retain_grad()" a la variable "embeddings" para conservar su gradiente durante la retropropagación.

Aplicamos el método ".sum()" a la variable "estado_final" para obtener un escalar que represente la suma de todos los elementos del estado final.
La variable "estado_final_sum" contiene la suma de todos los elementos del estado final, convirtiéndolo en un escalar adecuado para la retropropagación.

Aplicamos el método ".backward()" a la variable "estado_final_sum" para calcular los gradientes respecto a los embeddings de entrada.
La variable "estado_final_sum" almacena un escalar que representa la suma de todos los elementos del estado final.

Utilizamos el atributo ".grad" y la indexación "[]" de "embeddings" para obtener los gradientes calculados durante la retropropagación.
El rango "[:, 0]" selecciona el gradiente correspondiente al primer token de cada secuencia en el batch.
La variable "gradiente_inicial" almacena el tensor del gradiente correspondiente al primer token de cada secuencia en el batch.
Este gradiente puede ser muy pequeño, pero una sola medición depende de la inicialización y no demuestra por sí sola el desvanecimiento."""

embeddings.retain_grad()

estado_final_sum = estado_final.sum()

estado_final_sum.backward()

gradiente_inicial = embeddings.grad[:, 0]

gradiente_inicial # Muestra el gradiente correspondiente al primer token.


In [ ]:
# Bloque 6 — Definición de una LSTM para comparar estabilidad
"""Este bloque define una LSTM que incorpora mecanismos de puertas para mitigar el desvanecimiento del gradiente.

Aplicamos la clase "nn.LSTM()" a la variable "lstm" para definir una red neuronal recurrente con memoria mejorada.
El parámetro "input_size=" indica la dimensionalidad de los embeddings de entrada, en este caso el valor de "embedding_dim", es decir, 32 dimensiones.
El parámetro "hidden_size=" define la dimensionalidad del estado oculto, en este caso el valor de "hidden_size", es decir, 64 dimensiones.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".

La variable "lstm" almacena una arquitectura recurrente diseñada para facilitar el aprendizaje de dependencias largas.
La LSTM incorpora mecanismos de puertas que regulan el flujo de información a través de la red, permitiendo retener o descartar 
información según sea necesario."""

lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, batch_first=True)

lstm # Muestra la arquitectura de la LSTM definida.


In [ ]:
# Bloque 7 — Forward pass con LSTM
"""Este bloque procesa la secuencia mediante la LSTM.
Aplicamos la función "torch.zeros()" a las variables "estado_oculto" y "estado_celda", respectivamente, para inicializar el estado oculto 
y el estado de celda con ceros en todas sus dimensiones.
        
El argumento "1" en cada caso indica el número de capas de la LSTM.
El segundo argumento "1" en cada caso indica el tamaño del batch, es decir, el número de secuencias que se procesarán simultáneamente.
El argumento "hidden_size" en cada caso indica la dimensionalidad del estado oculto de la LSTM y, por lo tanto, también del estado de celda.
Las variables "estado_oculto" y "estado_celda" almacenan el estado oculto inicial y el estado de celda inicial de la LSTM, 
que en este caso son tensores de ceros en todas sus dimensiones (1, 1, hidden_size).

Asignamos de forma múltiple la llamada a la instancia de la clase "LSTM" a las variables "salida_lstm" y "estados_finales" mediante 
la expresión "lstm(embeddings, (estado_oculto, estado_celda))".
El argumento "embeddings" es el conjunto de tensores densos de entrada que queremos procesar mediante la LSTM.
El argumento "estado_oculto" es el estado oculto de partida definido previamente para la LSTM.
El argumento "estado_celda" es el estado de celda de partida definido previamente para la LSTM.
En este caso, se pasan los estados en forma de tupla "()", donde el primer elemento es el estado oculto y el segundo elemento es el estado de celda.

La variable "salida_lstm" almacena las representaciones generadas por la LSTM en cada paso de la secuencia.
La variable "estados_finales" almacena los estados oculto y de celda resultantes de la LSTM después de procesar toda la secuencia.
Este bloque solo ejecuta la LSTM: no calcula su gradiente ni compara su estabilidad con la RNN."""

estado_oculto = torch.zeros(1, 1, hidden_size)
estado_celda = torch.zeros(1, 1, hidden_size)

salida_lstm, estados_finales = lstm(embeddings, (estado_oculto, estado_celda))

salida_lstm, estados_finales # Muestra la salida de la LSTM y los estados finales (estado oculto y estado de celda).


In [ ]:
# Bloque 8 — Medición del coste computacional
"""Este bloque mide el tiempo de procesamiento de la secuencia mediante la RNN y la LSTM.
Este bloque importa "time" para trabajar con mediciones de tiempo en Python.

Aplicamos la función "time.time()" a la variable "inicio_rnn" para registrar el tiempo de inicio del procesamiento de la RNN.
La variable "inicio_rnn" almacena el tiempo de inicio del procesamiento de la RNN.

Llamamos a la instancia de la clase RNN mediante la expresión "rnn(embeddings, estado_inicial)" para procesar la secuencia con el estado 
inicial definido previamente.
Los argumentos de la función "rnn" son los embeddings de entrada y el estado inicial de la RNN.

Aplicamos la función "time.time()" a la variable "tiempo_rnn" para registrar el tiempo de finalización del procesamiento de la RNN.
Además, le restamos el tiempo de inicio para obtener el tiempo total de procesamiento de la RNN.
La variable "tiempo_rnn" almacena el tiempo requerido por la RNN.

Aplicamos la función "time.time()" a la variable "inicio_lstm" para registrar el tiempo de inicio del procesamiento de la LSTM.
La variable "inicio_lstm" almacena el tiempo de inicio del procesamiento de la LSTM.

Llamamos a la instancia de la clase LSTM mediante la expresión "lstm(embeddings, (estado_oculto, estado_celda))" para procesar la secuencia con 
los estados iniciales definidos previamente.
Los argumentos de la función "lstm" son los embeddings de entrada y una tupla con el estado oculto y el estado de celda inicial de la LSTM.

Aplicamos la función "time.time()" a la variable "tiempo_lstm" para registrar el tiempo de finalización del procesamiento de la LSTM.
Además, le restamos el tiempo de inicio para obtener el tiempo total de procesamiento de la LSTM.
Una única medición por modelo, sin repeticiones ni comparación con otras arquitecturas, no permite concluir su coste relativo ni demostrar la falta de paralelización."""

import time

inicio_rnn = time.time()
rnn(embeddings, estado_inicial)
tiempo_rnn = time.time() - inicio_rnn

inicio_lstm = time.time()
lstm(embeddings, (estado_oculto, estado_celda))
tiempo_lstm = time.time() - inicio_lstm

tiempo_rnn, tiempo_lstm # Muestra los tiempos de procesamiento de la RNN y la LSTM.


In [ ]:
# Bloque 9 — Resultados finales
"""Este bloque organiza los resultados del análisis de limitaciones de los modelos recurrentes.
La variable "resultado" almacena en forma de diccionario los resultados del análisis, incluyendo gradientes, tiempos de ejecución y estados finales.
La estructura muestra el gradiente de la RNN y tiempos orientativos, sin comparar gradientes ni establecer el coste relativo de las arquitecturas."""

resultado = {
    "gradiente_inicial": gradiente_inicial,
    "tiempo_rnn": tiempo_rnn,
    "tiempo_lstm": tiempo_lstm,
    "estado_final_rnn": estado_final,
    "estado_final_lstm": estados_finales[0]
}

resultado # Muestra los resultados finales del análisis de limitaciones de los modelos recurrentes.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto por qué los modelos recurrentes, aunque útiles, encuentran límites claros en problemas lingüísticos más exigentes.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Los cuellos de botella de una arquitectura suelen abrir la puerta a nuevas soluciones más escalables.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Seguir aumentando complejidad sobre una arquitectura sin reconocer primero sus límites estructurales.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en los Transformers para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en los Transformers y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">04_Transformers/01_Los_transformers.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">